# MACE-MP foundation model · GMD-26 force-error gallery (zero-shot)

The foundation-model twin of `simurgh_gmd_force_error_gallery.ipynb`: same five GMD-26
*Fragment Duplication* trajectories, same two-sphere force-error encoding, same metrics —
but the potential is a **universal** model (MACE-MP-0, trained on MPTrj) that has never
seen a carboxylic acid from this dataset, loaded through GOAL's own interfaces rather than
`mace.calculators` directly. Read the two notebooks side by side and the numbers line up.

**Kernel** — run this with the **`GOAL (mace-cuda, standalone)`** kernel
(`.venv-mace`). `mace-torch` pins `e3nn==0.4.4`, which is incompatible with the e3nn that
GOAL's own SIMURGH blocks need, so MACE lives in a separate venv (see the "MACE
(mace-torch)" note in `pyproject.toml`). The pixi `cuda` env cannot run this notebook, and
this venv cannot run the SIMURGH one.

**Two GOAL entry points, both used below:**

| route | what it gives you | used in |
|---|---|---|
| `BACKBONE_REGISTRY.get("mace_finetune")` → `MACEFinetune` | a GOAL backbone that eats `AtomicGraph` and returns `{energy, forces, num_atoms}` — batchable, fine-tunable with `head: null` | §1–§6 |
| `CalculatorFactory.create("mace", …)` | MACE's own ASE calculator, the `goal.md` MD path | §3, §7 |

Swap `"mace"` for `"upet"` in the factory and the same code drives a metatensor PET
foundation model (`pet-mad-s`, …) — `configs/md/calculators/upet.yaml` has the full config.
It needs `pip install upet`, which is *not* in this venv, so this notebook uses MACE.

**Encoding** (unchanged from the SIMURGH gallery): inner sphere = relative magnitude error
`|‖F_pred‖ − ‖F_ref‖| / ‖F_ref‖` (white → pink, 1st–99th percentile of that trajectory);
outer shell = cosine similarity `cos(F_pred, F_ref)` (white = aligned → red → yellow =
opposite), opacity ∝ directional error; every frame Kabsch-aligned to frame 0.

In [ ]:
import torch

# e3nn 0.4.4 (pinned by mace-torch) `torch.load`s a constants file containing a `slice`
# object, which torch >= 2.6 refuses under the new weights_only=True default. Allow-list it
# BEFORE anything imports e3nn. (Importing `mace` first also works — it sets
# TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 — but relying on import order is fragile.)
torch.serialization.add_safe_globals([slice])

print(f"torch {torch.__version__} · CUDA available: {torch.cuda.is_available()}")

In [ ]:
from __future__ import annotations

import io
import time
from pathlib import Path

import ase
import ase.io
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
from ase.neighborlist import NeighborList, natural_cutoffs
from matplotlib.cm import ScalarMappable
from torch_geometric.data import Batch

import ipywidgets as widgets
import py3Dmol
from IPython.display import HTML, display

import goal.ml.nn.models  # noqa: F401  — auto-discovery fires the registry decorators
from goal.ml.data.graph import AtomicGraph
from goal.ml.registry import BACKBONE_REGISTRY

import mace

print(f"mace-torch {mace.__version__ if hasattr(mace, '__version__') else '0.3.x'}")

## 0 · Configuration

`MODEL` is what GOAL hands to the MACE loader: `"small"` / `"medium"` / `"large"` pull the
corresponding MACE-MP release (cached under `~/.cache/mace/`), or give a path to a
`.model` file for a local / fine-tuned checkpoint.

The trajectory list is identical to the SIMURGH gallery on purpose — 2 in-distribution
monocarboxylic acids and 3 out-of-distribution dicarboxylic acids. "ID / OOD" here labels
the *GMD benchmark* splits, not this model's training distribution: MACE-MP saw neither
family, so it is zero-shot on all five.

In [ ]:
PROJECT_ROOT = Path("..").resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "GMD" / "FragmentDuplication"

MODEL = "medium"  # MACE-MP-0 medium (L1, r_max 6 Å, 89 elements)
DTYPE = "float64"  # MACE's own recommendation for anything gradient-based

TRAJECTORIES: list[tuple[str, str]] = [
    ("TestID", "O2C4H8_PBE.traj"),
    ("TestID", "O2C10H20_PBE.traj"),
    ("TestOOD", "O4C4H6_PBE.traj"),
    ("TestOOD", "O4C7H12_PBE.traj"),
    ("TestOOD", "O4C10H18_PBE.traj"),
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FRAME_STRIDE = 1  # >1 subsamples frames (each trajectory holds 200)
BATCH_SIZE = 4  # frames per forward pass; auto-halves on CUDA OOM
PLAYBACK_FPS = 3

assert DATA_ROOT.is_dir(), f"GMD data not found: {DATA_ROOT}"
print(f"data   : {DATA_ROOT}")
print(f"model  : {MODEL}  ({DTYPE})")
print(f"device : {DEVICE}")

## 1 · Load MACE through the GOAL backbone registry

`MACEFinetune` is a real `nn.Module` GOAL backbone: it converts an `AtomicGraph` into
MACE's input dict (one-hot `node_attrs` over the model's own z-table, Cartesian `shifts`
from `unit_shifts @ cell`), calls MACE with `compute_force=True`, and returns GOAL's
`{energy, forces, num_atoms}`. That is why the batched-inference code below is identical
to the SIMURGH notebook's, and why the same object could be fine-tuned on GMD with
`head: null` (`strategy="head_only" | "full" | "lora"`).

`strategy` only decides which parameters stay trainable, so it is irrelevant to inference;
`reestimate_e0s=False` keeps MACE's own MPTrj atomic-energy baseline — §3.5 measures what
that baseline costs against GMD's DFT settings instead of overwriting it.

In [ ]:
t0 = time.perf_counter()
MACEFinetune = BACKBONE_REGISTRY.get("mace_finetune")
backbone = MACEFinetune(
    checkpoint=MODEL,
    device=DEVICE,
    dtype=DTYPE,
    strategy="head_only",
    reestimate_e0s=False,
)
backbone = backbone.to(DEVICE).eval()
CUTOFF = float(backbone.cutoff)
TORCH_DTYPE = backbone.torch_dtype
print(f"\nloaded in {time.perf_counter() - t0:.1f}s")
print(f"backbone   : {type(backbone).__name__}  (registry key 'mace_finetune')")
print(f"parameters : {sum(p.numel() for p in backbone.parameters()):,}")
print(f"cutoff     : {CUTOFF} Å   (MACE r_max, read off the model)")
print(f"elements   : {len(backbone.elements)} in the z-table — "
      f"H,C,O present: {all(z in backbone.elements for z in (1, 6, 8))}")
print(f"dtype      : {TORCH_DTYPE}")

## 2 · Batched inference

Same shape as the SIMURGH gallery, one simplification: MACE needs no `fragment_index`, so
`AtomicGraph.from_ase` is called with nothing but the model's own cutoff. Note the graph is
built at MACE's `r_max = 6 Å`, not the 8 Å SIMURGH was trained with — each model gets the
neighbourhood it expects.

In [ ]:
def to_graph(atoms: ase.Atoms) -> AtomicGraph:
    """ASE frame → AtomicGraph at the foundation model's own cutoff."""
    return AtomicGraph.from_ase(atoms, cutoff=CUTOFF, dtype=TORCH_DTYPE)


def _forward(chunk: list[ase.Atoms]) -> tuple[np.ndarray, list[np.ndarray]]:
    """Energies + per-frame forces for one batch of frames."""
    batch = Batch.from_data_list([to_graph(a) for a in chunk]).to(DEVICE)
    # MACE differentiates its own energy w.r.t. positions inside forward, so grad must be on.
    with torch.enable_grad():
        out = backbone(batch)
    energies = out["energy"].detach().cpu().numpy()
    forces = out["forces"].detach().cpu().numpy()
    del out, batch
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return energies, np.split(forces, np.cumsum([len(a) for a in chunk])[:-1])


def predict_frames(
    frames: list[ase.Atoms], batch_size: int = BATCH_SIZE
) -> tuple[np.ndarray, list[np.ndarray]]:
    """Predict energies/forces for every frame, halving the batch size on CUDA OOM."""
    energies: list[np.ndarray] = []
    forces: list[np.ndarray] = []
    start, bs = 0, batch_size
    while start < len(frames):
        try:
            e, f = _forward(frames[start : start + bs])
        except torch.OutOfMemoryError:
            if bs == 1:
                raise
            bs = max(1, bs // 2)
            torch.cuda.empty_cache()
            print(f"  CUDA OOM — retrying with batch_size={bs}")
            continue
        energies.append(e)
        forces.extend(f)
        start += bs
    return np.concatenate(energies), forces


PREDICTIONS: dict[tuple[str, str], dict[str, object]] = {}


def evaluate_trajectory(split: str, molecule: str, stride: int = FRAME_STRIDE) -> dict[str, object]:
    """Load one trajectory, predict every frame, and derive the per-atom error fields."""
    key = (split, molecule)
    if key in PREDICTIONS:
        return PREDICTIONS[key]

    path = DATA_ROOT / split / molecule
    if not path.is_file():
        available = sorted(p.name for p in (DATA_ROOT / split).glob("*.traj"))
        raise FileNotFoundError(f"{path} not found. Available: {available}")

    frames = ase.io.read(str(path), index=f"::{stride}")
    ref_energy = np.array([f.get_potential_energy() for f in frames])
    ref_forces = [f.get_forces().copy() for f in frames]

    t_start = time.perf_counter()
    pred_energy, pred_forces = predict_frames(frames)
    elapsed = time.perf_counter() - t_start

    n_frames, n_atoms = len(frames), len(frames[0])
    rel_mag = np.zeros((n_frames, n_atoms))
    cos_sim = np.zeros((n_frames, n_atoms))
    for i, (pred, ref) in enumerate(zip(pred_forces, ref_forces)):
        norm_pred = np.linalg.norm(pred, axis=1)
        norm_ref = np.linalg.norm(ref, axis=1)
        denom = norm_pred * norm_ref
        cos_sim[i] = np.divide(
            (pred * ref).sum(axis=1), denom, out=np.zeros_like(denom), where=denom > 1e-20
        )
        rel_mag[i] = np.abs(norm_pred - norm_ref) / (norm_ref + 1e-10)

    record: dict[str, object] = {
        "split": split,
        "molecule": molecule,
        "formula": frames[0].get_chemical_formula(),
        "symbols": frames[0].get_chemical_symbols(),
        "numbers": frames[0].get_atomic_numbers(),
        "n_frames": n_frames,
        "n_atoms": n_atoms,
        "positions": np.array([f.get_positions() for f in frames]),
        "ref_energy": ref_energy,
        "pred_energy": pred_energy,
        "ref_forces": ref_forces,
        "pred_forces": pred_forces,
        "rel_mag": rel_mag,
        "cos_sim": cos_sim,
        "forces_mae": float(np.mean([np.abs(p - r).mean() for p, r in zip(pred_forces, ref_forces)])),
        "seconds": elapsed,
    }
    PREDICTIONS[key] = record
    print(
        f"{split}/{molecule}: {n_frames} frames × {n_atoms} atoms in {elapsed:.1f}s "
        f"({elapsed / n_frames * 1e3:.0f} ms/frame)"
    )
    return record

## 3 · Sanity check — one frame, and does the GOAL adapter agree with MACE itself?

`MACEFinetune` builds MACE's input dict by hand from an `AtomicGraph`, so the first thing
to verify is that it reproduces what MACE's own ASE calculator computes for the same frame.
The second calculator comes from `goal.md`'s factory — the MD-side GOAL interface — so this
cell checks both routes at once. (The factory defaults to float32 while the backbone runs
float64, which is the whole size of the residual below.)

In [ ]:
from goal.md.core.calculator_factory import CalculatorFactory

probe = ase.io.read(str(DATA_ROOT / "TestID" / "O2C6H12_PBE.traj"), index=100)
probe_ref_e = probe.get_potential_energy()
probe_ref_f = probe.get_forces().copy()
probe_pred_e, (probe_pred_f,) = predict_frames([probe])

factory_calc = CalculatorFactory.create("mace", model_name_or_path=MODEL, device=DEVICE)
probe_ase = probe.copy()
probe_ase.calc = factory_calc
factory_e = probe_ase.get_potential_energy()
factory_f = probe_ase.get_forces()

norm_pred = np.linalg.norm(probe_pred_f, axis=1)
norm_ref = np.linalg.norm(probe_ref_f, axis=1)
probe_cos = (probe_pred_f * probe_ref_f).sum(axis=1) / (norm_pred * norm_ref + 1e-12)

print(f"\n{probe.get_chemical_formula()} · frame 100 · {len(probe)} atoms\n")
print("GOAL backbone vs MACE's own calculator (CalculatorFactory):")
print(f"  max |ΔF| = {np.abs(probe_pred_f - factory_f).max():.2e} eV/Å"
      f"     |ΔE| = {abs(probe_pred_e[0] - factory_e):.2e} eV")
print("\nvs DFT reference:")
print(f"  {'energy (eV)':24s} DFT {probe_ref_e:14.4f}   MACE {probe_pred_e[0]:12.4f}")
print(f"  {'raw ΔE (eV)':24s}     {probe_pred_e[0] - probe_ref_e:14.4f}"
      f"   ← different reference/basis, see §3.5")
print(f"  {'forces MAE (eV/Å)':24s}     {np.abs(probe_pred_f - probe_ref_f).mean():14.5f}")
print(f"  {'mean cos(F_pred,F_ref)':24s}     {probe_cos.mean():14.5f}")
print(f"  {'net force ‖ΣF‖ (eV/Å)':24s} DFT {np.linalg.norm(probe_ref_f.sum(axis=0)):14.2e}"
      f"   MACE {np.linalg.norm(probe_pred_f.sum(axis=0)):12.2e}")

## 3.5 · Why the energies cannot be compared raw

MACE-MP-0's total energies are on MPTrj's scale (VASP PAW/PBE); GMD's references are
molecular PBE/def2-SVP. The gap is thousands of eV, and it is very nearly a **sum of
per-element constants** — exactly the E0 baseline that
`goal.ml.nn.models.foundation.base.estimate_reference_energies` re-fits before fine-tuning.

So the honest zero-shot energy metric is the error *after* removing that baseline: fit
`E_DFT − E_MACE ≈ Σ_Z n_Z · Δe_Z` by least squares over every frame of all five
trajectories (3 unknowns for H, C, O; ~1000 rows), then report the residual. It is fitted
on the same frames it is scored on — a 3-parameter alignment, not a held-out claim.
Forces need none of this: they are unaffected by any per-element constant.

In [ ]:
def fit_element_baseline(records: list[dict]) -> dict[int, float]:
    """Least-squares per-element offsets Δe_Z from  E_DFT − E_pred ≈ Σ_Z n_Z Δe_Z."""
    elements = sorted({int(z) for r in records for z in set(r["numbers"])})
    design, residual = [], []
    for r in records:
        counts = np.array([[np.sum(np.asarray(r["numbers"]) == z) for z in elements]], dtype=float)
        design.append(np.repeat(counts, r["n_frames"], axis=0))
        residual.append(np.asarray(r["ref_energy"]) - np.asarray(r["pred_energy"]))
    offsets, *_ = np.linalg.lstsq(np.vstack(design), np.concatenate(residual), rcond=None)
    return {z: float(o) for z, o in zip(elements, offsets)}


def aligned_energy(record: dict, baseline: dict[int, float]) -> np.ndarray:
    """`pred_energy` shifted onto the DFT scale using the fitted per-element offsets."""
    shift = float(sum(baseline[int(z)] for z in np.asarray(record["numbers"])))
    return np.asarray(record["pred_energy"]) + shift


print("evaluating all five trajectories (cached for §5 and §6) ...")
rows = [evaluate_trajectory(split, molecule) for split, molecule in TRAJECTORIES]
BASELINE = fit_element_baseline(rows)

print("\nfitted per-element offsets  Δe_Z  (eV/atom):")
for z, offset in BASELINE.items():
    print(f"  Z={z:2d}  {offset:+12.4f}")
print("\nper-trajectory energy error, raw vs baseline-aligned:")
print(f"  {'molecule':10s} {'raw |ΔE|/atom':>14s} {'aligned |ΔE|/atom':>18s}")
for r in rows:
    raw = np.abs(np.asarray(r["pred_energy"]) - np.asarray(r["ref_energy"])).mean() / r["n_atoms"]
    r["energy_mae_per_atom"] = float(
        np.abs(aligned_energy(r, BASELINE) - np.asarray(r["ref_energy"])).mean() / r["n_atoms"]
    )
    print(f"  {r['formula']:10s} {raw:14.3f} {r['energy_mae_per_atom']:18.5f}")

## 4 · Viewer

Byte-for-byte the SIMURGH gallery's viewer, so the two notebooks are visually comparable.
The magnitude colour scale is per-trajectory (1st–99th percentile), which auto-adapts to
this model's larger errors — read the colourbar, not the colour, when comparing notebooks.

In [ ]:
CMAP_MAG = mcolors.LinearSegmentedColormap.from_list("mag_err", ["#ffffff", "#ffb6d5", "#ff1493"])
CMAP_DIR = mcolors.LinearSegmentedColormap.from_list(
    "dir_err", [(0.0, "#ffffff"), (0.5, "#e60026"), (1.0, "#ffd000")]
)
CMAP_DIR_CB = mcolors.LinearSegmentedColormap.from_list(
    "dir_err_cb", [(0.0, "#ffd000"), (0.5, "#e60026"), (1.0, "#ffffff")]
)
OUTER_OPACITY_MIN, OUTER_OPACITY_MAX = 0.25, 0.85
VIEWER_WIDTH, VIEWER_HEIGHT = 600, 460


def kabsch_align(mobile: np.ndarray, ref: np.ndarray) -> np.ndarray:
    """Rigidly rotate + translate `mobile` (n_atoms, 3) onto `ref`, minimising RMSD."""
    mobile_com = mobile.mean(axis=0)
    ref_com = ref.mean(axis=0)
    H = (mobile - mobile_com).T @ (ref - ref_com)
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1, 1, d]) @ U.T
    return (R @ (mobile - mobile_com).T).T + ref_com


def colorbar_figure(norm, cmap, label):
    fig, ax = plt.subplots(figsize=(1.0, 3.6))
    fig.subplots_adjust(left=0.45, right=0.6, top=0.92, bottom=0.1)
    fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), cax=ax, orientation="vertical").set_label(
        label, fontsize=8
    )
    plt.close(fig)
    return fig


def show_trajectory(split: str, molecule: str) -> None:
    """Render the interactive force-error playback for one GMD trajectory."""
    record = evaluate_trajectory(split, molecule)
    symbols: list[str] = record["symbols"]  # type: ignore[assignment]
    n_atoms: int = record["n_atoms"]  # type: ignore[assignment]
    n_frames: int = record["n_frames"]  # type: ignore[assignment]
    rel_mag: np.ndarray = record["rel_mag"]  # type: ignore[assignment]
    cos_sim: np.ndarray = record["cos_sim"]  # type: ignore[assignment]

    positions = np.array(record["positions"])
    reference = positions[0].copy()
    positions = np.array([kabsch_align(p, reference) for p in positions])

    mag_vmin, mag_vmax = np.percentile(rel_mag, [1, 99])
    if mag_vmax <= mag_vmin:
        mag_vmax = mag_vmin + 1e-6

    # Energy fluctuation about this trajectory's own mean — the constant baseline of §3.5
    # would otherwise swamp the per-frame signal.
    d_energy = np.asarray(record["pred_energy"]) - np.asarray(record["ref_energy"])
    d_energy = d_energy - d_energy.mean()

    def two_sphere_colors(frame_mag: np.ndarray, frame_cos: np.ndarray):
        inner = [mcolors.to_hex(CMAP_MAG(min(float(m) / mag_vmax, 1.0))) for m in frame_mag]
        t_dir = (1.0 - frame_cos) / 2.0
        outer = [mcolors.to_hex(CMAP_DIR(t)) for t in t_dir]
        opacity = OUTER_OPACITY_MIN + t_dir * (OUTER_OPACITY_MAX - OUTER_OPACITY_MIN)
        return inner, outer, opacity

    first_frame = ase.Atoms(symbols=symbols, positions=positions[0])
    nl = NeighborList(natural_cutoffs(first_frame, mult=1.1), self_interaction=False, bothways=False)
    nl.update(first_frame)
    bonds = [(i, j) for i in range(n_atoms) for j in nl.get_neighbors(i)[0] if j > i]

    oxygen_idx = [i for i, s in enumerate(symbols) if s == "O"]
    first_carbon_idx = next((i for i, s in enumerate(symbols) if s == "C"), None)
    tracked_labels = {idx: f"O{n + 1}" for n, idx in enumerate(oxygen_idx)}
    if first_carbon_idx is not None:
        tracked_labels[first_carbon_idx] = "C1"

    def build_view(frame_idx: int) -> py3Dmol.view:
        frame_pos = positions[frame_idx]
        inner, outer, opacity = two_sphere_colors(rel_mag[frame_idx], cos_sim[frame_idx])

        buf = io.StringIO()
        ase.io.write(buf, ase.Atoms(symbols=symbols, positions=frame_pos), format="xyz")

        view = py3Dmol.view(width=VIEWER_WIDTH, height=VIEWER_HEIGHT)
        view.addModel(buf.getvalue(), "xyz")
        view.setStyle({}, {})  # suppress py3Dmol's own bonds; we draw them below

        for i, j in bonds:
            p1, p2 = frame_pos[i], frame_pos[j]
            view.addCylinder({
                "start": {"x": float(p1[0]), "y": float(p1[1]), "z": float(p1[2])},
                "end": {"x": float(p2[0]), "y": float(p2[1]), "z": float(p2[2])},
                "radius": 0.08,
                "color": "#888888",
                "fromCap": True,
                "toCap": True,
            })

        for i, pos in enumerate(frame_pos):
            x, y, z = float(pos[0]), float(pos[1]), float(pos[2])
            view.addSphere({"center": {"x": x, "y": y, "z": z}, "radius": 0.42,
                            "color": outer[i], "opacity": float(opacity[i])})
            view.addSphere({"center": {"x": x, "y": y, "z": z}, "radius": 0.22,
                            "color": inner[i]})

        for i, (sym, pos) in enumerate(zip(symbols, frame_pos)):
            view.addLabel(tracked_labels.get(i, sym), {
                "position": {"x": float(pos[0]), "y": float(pos[1]), "z": float(pos[2])},
                "fontSize": 12 if i in tracked_labels else 11,
                "fontColor": "black",
                "showBackground": False,
                "alignment": "center",
            })

        view.zoomTo()
        return view

    viewer_out, info_out, err_out = widgets.Output(), widgets.Output(), widgets.Output()
    cb_mag_out, cb_dir_out = widgets.Output(), widgets.Output()
    with cb_mag_out:
        display(colorbar_figure(mcolors.Normalize(0.0, float(mag_vmax)), CMAP_MAG,
                                "Inner — relative |F| error"))
    with cb_dir_out:
        display(colorbar_figure(mcolors.Normalize(-1.0, 1.0), CMAP_DIR_CB,
                                "Outer — cosine similarity"))

    err_fig, err_ax = plt.subplots(figsize=(9.5, 2.0))
    err_ax.plot(rel_mag.mean(axis=1), color="darkmagenta")
    err_ax.set_xlim(0, n_frames - 1)
    err_ax.set_xlabel("frame", fontsize=9)
    err_ax.set_ylabel("|F| error", color="darkmagenta", fontsize=9)
    err_ax.tick_params(axis="y", labelcolor="darkmagenta", labelsize=8)
    err_ax.tick_params(axis="x", labelsize=8)
    err_ax2 = err_ax.twinx()
    err_ax2.plot(cos_sim.mean(axis=1), color="firebrick", alpha=0.7)
    err_ax2.set_ylabel("cos sim", color="firebrick", fontsize=9)
    err_ax2.tick_params(axis="y", labelcolor="firebrick", labelsize=8)
    err_ax2.set_ylim(-1.05, 1.05)
    err_marker = err_ax.axvline(0, color="black", linewidth=1.2)
    err_fig.tight_layout()
    plt.close(err_fig)

    def render_frame(frame_idx: int) -> None:
        view = build_view(frame_idx)
        viewer_out.clear_output(wait=True)
        with viewer_out:
            view.show()

        err_marker.set_xdata([frame_idx, frame_idx])
        err_out.clear_output(wait=True)
        with err_out:
            display(err_fig)

        mag, cos = rel_mag[frame_idx], cos_sim[frame_idx]
        info_out.clear_output(wait=True)
        with info_out:
            display(HTML(
                f"<div style='font-family:sans-serif;font-size:13px;line-height:1.7;padding-left:8px'>"
                f"<b>{record['formula']}<br>{split} · {Path(molecule).stem}</b><br>"
                f"model = MACE-MP-0 ({MODEL})<br>"
                f"cutoff = {CUTOFF} Å · zero-shot<br><br>"
                f"frame = {frame_idx}/{n_frames - 1}<br>"
                f"mean |F| err&nbsp;= {mag.mean():.4f}<br>"
                f"max&nbsp;|F| err&nbsp;= {mag.max():.4f}<br>"
                f"mean cos sim&nbsp;= {cos.mean():.4f}<br>"
                f"min&nbsp;cos sim&nbsp;= {cos.min():.4f}<br>"
                f"ΔE − ⟨ΔE⟩ = {d_energy[frame_idx]:+.4f} eV<br>"
                f"atoms = {n_atoms}"
                f"</div>"
            ))

    slider = widgets.IntSlider(min=0, max=n_frames - 1, step=1, value=0, description="frame")
    play = widgets.Play(min=0, max=n_frames - 1, step=1, interval=int(1000 / PLAYBACK_FPS))
    widgets.jslink((play, "value"), (slider, "value"))
    slider.observe(lambda change: render_frame(change["new"]), names="value")

    render_frame(0)
    display(widgets.HBox([play, slider]))
    display(widgets.HBox([viewer_out, cb_mag_out, cb_dir_out, info_out]))
    display(err_out)


print("viewer ready")

## 5 · The gallery

Five trajectories, five viewers, predictions already cached by §3.5 — pressing **▶** only
redraws. Watch the outer shells: a foundation model's directional errors are spread over
every atom rather than concentrated on the functional group.

### GMD in-distribution split — monocarboxylic acids

#### O2C4H8

In [ ]:
show_trajectory("TestID", "O2C4H8_PBE.traj")

#### O2C10H20

In [ ]:
show_trajectory("TestID", "O2C10H20_PBE.traj")

### GMD out-of-distribution split — dicarboxylic acids

#### O4C4H6

In [ ]:
show_trajectory("TestOOD", "O4C4H6_PBE.traj")

#### O4C7H12

In [ ]:
show_trajectory("TestOOD", "O4C7H12_PBE.traj")

#### O4C10H18

In [ ]:
show_trajectory("TestOOD", "O4C10H18_PBE.traj")

## 6 · Summary, and how it compares to the trained SIMURGH

Energies are baseline-aligned per §3.5; forces are raw. The SIMURGH column is the
`simurgh_gmd26` run measured in `simurgh_gmd_force_error_gallery.ipynb` on these exact
trajectories (2026-08-05) — quoted here for context, not recomputed (its e3nn version
cannot be imported in this venv).

In [ ]:
header = (f"{'split':8s} {'molecule':10s} {'atoms':>5s} {'frames':>6s} "
          f"{'E MAE/atom*':>12s} {'F MAE':>9s} {'F max err':>9s} {'mean cos':>9s} {'min cos':>8s}")
print(header)
print("-" * len(header))
for r in rows:
    max_component = max(float(np.abs(p - q).max()) for p, q in zip(r["pred_forces"], r["ref_forces"]))
    print(f"{r['split']:8s} {r['formula']:10s} {r['n_atoms']:5d} {r['n_frames']:6d} "
          f"{r['energy_mae_per_atom']:12.5f} {r['forces_mae']:9.5f} {max_component:9.4f} "
          f"{np.mean(r['cos_sim']):9.5f} {np.min(r['cos_sim']):8.4f}")
print("\n* baseline-aligned (§3.5). units: eV/atom, eV/Å, eV/Å")

SIMURGH_FORCES_MAE = {  # measured 2026-08-05, simurgh_gmd_force_error_gallery.ipynb
    "C4H8O2": 0.01225, "C10H20O2": 0.01297,
    "C4H6O4": 1.31250, "C7H12O4": 0.22193, "C10H18O4": 0.51223,
}
print(f"\n{'molecule':10s} {'MACE-MP F MAE':>14s} {'SIMURGH F MAE':>14s} {'ratio':>8s}")
for r in rows:
    ref = SIMURGH_FORCES_MAE[r["formula"]]
    print(f"{r['formula']:10s} {r['forces_mae']:14.5f} {ref:14.5f} "
          f"{r['forces_mae'] / ref:8.2f}×")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for r in rows:
    style = "-" if r["split"] == "TestID" else "--"
    axes[0].plot(r["rel_mag"].mean(axis=1), style, linewidth=1.2, label=f"{r['split']} {r['formula']}")
    axes[1].plot(r["cos_sim"].mean(axis=1), style, linewidth=1.2)
axes[0].set_xlabel("frame")
axes[0].set_ylabel("mean relative |F| error")
axes[0].set_title(f"MACE-MP-0 ({MODEL}) — force-magnitude error per frame")
axes[0].grid(alpha=0.3)
axes[1].set_xlabel("frame")
axes[1].set_ylabel("mean cos(F_pred, F_ref)")
axes[1].set_title("Force-direction agreement per frame")
axes[1].grid(alpha=0.3)
axes[0].legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()

## 7 · The same model as an ASE calculator

Two one-liners, both GOAL:

* `GOALCalculator(module=backbone, cutoff=…)` — wraps the backbone we already loaded.
  `MACEFinetune` returns exactly the `{energy, forces}` dict the calculator expects, so no
  subclassing is needed here (unlike the fragment-aware SIMURGH model, which needed
  `fragment_index` bolted onto graph construction).
* `CalculatorFactory.create("mace" | "upet", …)` — the `goal.md` route, which hands back
  the upstream package's own ASE calculator. This is what `configs/md/calculators/*.yaml`
  drives, so anything under `goal.md` (MTS/RESPA, i-PI, online learning) takes a foundation
  model by editing one YAML key.

In [ ]:
from ase.optimize import BFGS

from goal.ml.utils.calculator import GOALCalculator

ase_calc = GOALCalculator(module=backbone, cutoff=CUTOFF, device=DEVICE, dtype=TORCH_DTYPE)

relaxed = ase.io.read(str(DATA_ROOT / "TestOOD" / "O4C10H18_PBE.traj"), index=0)
relaxed.calc = ase_calc
e_start = relaxed.get_potential_energy()
f_start = np.abs(relaxed.get_forces()).max()

opt = BFGS(relaxed, logfile=None)
converged = opt.run(fmax=0.05, steps=25)

print(f"{relaxed.get_chemical_formula()} · MACE-MP-driven BFGS relaxation")
print(f"  start : E = {e_start:12.4f} eV   max|F| = {f_start:.4f} eV/Å")
print(f"  end   : E = {relaxed.get_potential_energy():12.4f} eV   "
      f"max|F| = {np.abs(relaxed.get_forces()).max():.4f} eV/Å")
print(f"  ΔE = {relaxed.get_potential_energy() - e_start:+.4f} eV over {opt.nsteps} steps "
      f"(converged: {converged})")

# The md-side route, for reference — same weights, upstream calculator:
#   calc = CalculatorFactory.create("mace", model_name_or_path="medium", device="cuda")
#   calc = CalculatorFactory.create("upet", model="pet-mad-s", device="cuda")  # needs `pip install upet`

---

### What to take away

| | |
|---|---|
| Model | MACE-MP-0 medium (L1), 4.7 M params, `r_max` 6 Å, 89 elements, float64 |
| Loaded via | `BACKBONE_REGISTRY.get("mace_finetune")` → `MACEFinetune` (§1); `CalculatorFactory.create("mace", …)` (§3, §7) |
| Verified | the GOAL adapter's hand-built MACE input matches MACE's own calculator to float32 round-off (§3) |
| Energies | only meaningful after a per-element baseline fit — MPTrj PBE/PAW vs GMD PBE/def2-SVP (§3.5) |
| Data | `data/GMD/FragmentDuplication/{TestID,TestOOD}`, the same 5 trajectories as the SIMURGH gallery |
| Kernel | `GOAL (mace-cuda, standalone)` — mace-torch pins e3nn 0.4.4 |

**Next steps this notebook sets up.** `MACEFinetune` is the fine-tuning object, not just an
inference wrapper: pass it as `model.backbone` with `head: null`, `strategy: head_only`,
and `reestimate_e0s: true` (the `FoundationE0Callback` then re-baselines E0 from the
training set — see `configs/ml/finetune/`) to turn the zero-shot numbers above into a
fine-tuned baseline on the same split. `PREDICTIONS` and `BASELINE` hold everything
computed here, keyed by `(split, molecule)`.